In [1]:
from sklearn.preprocessing import LabelEncoder

In [2]:
from datetime import datetime

In [3]:
import pandas as pd

# 导入数据
df = pd.read_csv('train .csv')

# 查看前5行
df.head()

,id,loanAmnt,term,interestRate,installment,grade,subGrade,employmentTitle,employmentLength,homeOwnership,...,n5,n6,n7,n8,n9,n10,n11,n12,n13,n14
0,0,35000.0,5,19.52,917.97,E,E2,320.0,2 years,2,...,9.0,8.0,4.0,12.0,2.0,7.0,0.0,0.0,0.0,2.0
1,1,18000.0,5,18.49,461.90,D,D2,219843.0,5 years,0,...,NaN,NaN,NaN,NaN,NaN,13.0,NaN,NaN,NaN,NaN
2,2,12000.0,5,16.99,298.17,D,D3,31698.0,8 years,0,...,0.0,21.0,4.0,5.0,3.0,11.0,0.0,0.0,0.0,4.0
3,3,11000.0,3,7.26,340.96,A,A4,46854.0,10+ years,1,...,16.0,4.0,7.0,21.0,6.0,9.0,0.0,0.0,0.0,1.0
4,4,3000.0,3,12.99,101.07,C,C2,54.0,NaN,1,...,4.0,9.0,10.0,15.0,7.0,12.0,0.0,0.0,0.0,4.0


In [4]:
print(df.columns.tolist())

['id', 'loanAmnt', 'term', 'interestRate', 'installment', 'grade', 'subGrade', 'employmentTitle', 'employmentLength', 'homeOwnership', 'annualIncome', 'verificationStatus', 'issueDate', 'isDefault', 'purpose', 'postCode', 'regionCode', 'dti', 'delinquency_2years', 'ficoRangeLow', 'ficoRangeHigh', 'openAcc', 'pubRec', 'pubRecBankruptcies', 'revolBal', 'revolUtil', 'totalAcc', 'initialListStatus', 'applicationType', 'earliesCreditLine', 'title', 'policyCode', 'n0', 'n1', 'n2', 'n3', 'n4', 'n5', 'n6', 'n7', 'n8', 'n9', 'n10', 'n11', 'n12', 'n13', 'n14']


In [5]:
# 查看缺失值

missing = df.isnull().sum()

missing = missing[missing > 0]

print("存在缺失值的变量：")
print(missing.sort_values(ascending=False))

存在缺失值的变量：
n11                   46537
employmentLength      31260
n14                   26950
n8                    26950
n2                    26950
n3                    26950
n5                    26950
n6                    26950
n0                    26950
n7                    26950
n9                    26950
n12                   26950
n13                   26950
n1                    26950
n10                   22229
n4                    22229
revolUtil               353
pubRecBankruptcies      265
dti                     154
postCode                  2
title                     2
employmentTitle           1
policyCode                1
applicationType           1
initialListStatus         1
totalAcc                  1
revolBal                  1
pubRec                    1
openAcc                   1
ficoRangeHigh             1
ficoRangeLow              1
delinquency_2years        1
regionCode                1
purpose                   1
isDefault                 1
earliesCre

In [6]:


df = df.dropna().reset_index(drop=True)

print("删除缺失值后维度:", df.shape)
print("剩余缺失值数量:", df.isnull().sum().sum())

删除缺失值后维度: (456752, 47)
剩余缺失值数量: 0


In [7]:
# =========================
# 4. 删除 id 和 policyCode
# =========================

drop_cols = ["id", "policyCode"]

df = df.drop(
    columns=[col for col in drop_cols if col in df.columns]
)

print("删除 id 和 policyCode 后维度:", df.shape)

删除 id 和 policyCode 后维度: (456752, 45)


In [10]:
# =========================
# 5. grade 和 subGrade 编码
# =========================

for col in ["grade", "subGrade"]:
    if col in df.columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))

print(df[["grade", "subGrade"]].head())

   grade  subGrade
0      4        21
1      3        17
2      0         3
3      0         4
4      0         3


In [11]:
# =========================
# 6. employmentLength 转数值
# =========================

def employment_to_int(x):
    x = str(x)

    if "< 1 year" in x:
        return 0
    elif "10+ years" in x:
        return 10
    else:
        return int(x.split()[0])

if "employmentLength" in df.columns:
    df["employmentLength"] = df["employmentLength"].apply(employment_to_int)

print(df["employmentLength"].value_counts().sort_index())

employmentLength
0      38963
1      31667
2      43821
3      38587
4      28669
5      29785
6      22299
7      21458
8      22022
9      18493
10    160988
Name: count, dtype: int64


In [17]:
from datetime import datetime

In [21]:
print(datetime)

<class 'datetime.datetime'>


In [25]:
from datetime import datetime

df['issueDate'] = pd.to_datetime(
    df['issueDate'],
    format='%Y-%m-%d'
)

startdate = datetime.strptime(
    '2007-06-01',
    '%Y-%m-%d'
)

df['issueDateDT'] = (
    df['issueDate'] - startdate
).dt.days

df[['issueDate','issueDateDT']].head()

,issueDate,issueDateDT
0,2014-07-01,2587
1,2015-10-01,3044
2,2015-08-01,2983
3,2017-04-01,3592
4,2014-10-01,2679


In [27]:
# =========================
# 8. earliesCreditLine 提取年份
# =========================

if "earliesCreditLine" in df.columns:
    df["earliesCreditLine"] = (
        df["earliesCreditLine"]
        .astype(str)
        .apply(lambda s: int(s[-4:]))
    )

print(df["earliesCreditLine"].head())

0    2001
1    2006
2    1999
3    1998
4    2006
Name: earliesCreditLine, dtype: int64


In [31]:
import numpy as np

In [33]:
# =========================
# 9. 查看非数值型变量
# =========================

non_numeric_cols = df.select_dtypes(
    exclude=np.number
).columns.tolist()

print("当前非数值型变量:")
print(non_numeric_cols)

当前非数值型变量:
['issueDate']


In [35]:
# =========================
# 11. 检查是否全是数值型变量
# =========================

non_numeric_cols = df.select_dtypes(
    exclude=np.number
).columns.tolist()

print("非数值型变量:")
print(non_numeric_cols)

print("数据类型:")
print(df.dtypes)

非数值型变量:
['issueDate']
数据类型:
loanAmnt                     float64
term                           int64
interestRate                 float64
installment                  float64
grade                          int32
subGrade                       int32
employmentTitle              float64
employmentLength               int64
homeOwnership                  int64
annualIncome                 float64
verificationStatus             int64
issueDate             datetime64[ns]
isDefault                    float64
purpose                      float64
postCode                     float64
regionCode                   float64
dti                          float64
delinquency_2years           float64
ficoRangeLow                 float64
ficoRangeHigh                float64
openAcc                      float64
pubRec                       float64
pubRecBankruptcies           float64
revolBal                     float64
revolUtil                    float64
totalAcc                     float64
initialLis

In [37]:
# =========================
# 12. 选择连续型变量
# 唯一值数量 > 15
# 不对目标变量 isDefault 做异常值处理
# =========================

target = "isDefault"

continuous_cols = []

for col in df.columns:
    if col != target and df[col].nunique() > 15:
        continuous_cols.append(col)

print("连续型变量数量:", len(continuous_cols))
print(continuous_cols)

连续型变量数量: 34
['loanAmnt', 'interestRate', 'installment', 'subGrade', 'employmentTitle', 'annualIncome', 'issueDate', 'postCode', 'regionCode', 'dti', 'delinquency_2years', 'ficoRangeLow', 'ficoRangeHigh', 'openAcc', 'pubRec', 'revolBal', 'revolUtil', 'totalAcc', 'earliesCreditLine', 'title', 'n0', 'n1', 'n2', 'n3', 'n4', 'n5', 'n6', 'n7', 'n8', 'n9', 'n10', 'n13', 'n14', 'issueDateDT']


In [39]:
# =========================
# 13. 使用 3-sigma 原则删除连续变量异常值
# =========================

print("3-sigma 删除前维度:", df.shape)

for col in continuous_cols:
    mean = df[col].mean()
    std = df[col].std()

    lower = mean - 3 * std
    upper = mean + 3 * std

    before_rows = df.shape[0]

    df = df[
        (df[col] >= lower) &
        (df[col] <= upper)
    ]

    after_rows = df.shape[0]

    print(
        col,
        "删除异常值数量:",
        before_rows - after_rows
    )

df = df.reset_index(drop=True)

print("3-sigma 删除后维度:", df.shape)

3-sigma 删除前维度: (456752, 46)
loanAmnt 删除异常值数量: 0
interestRate 删除异常值数量: 3414
installment 删除异常值数量: 4235
subGrade 删除异常值数量: 1620
employmentTitle 删除异常值数量: 1338
annualIncome 删除异常值数量: 3155
issueDate 删除异常值数量: 0
postCode 删除异常值数量: 572
regionCode 删除异常值数量: 0
dti 删除异常值数量: 1038
delinquency_2years 删除异常值数量: 12154
ficoRangeLow 删除异常值数量: 6201
ficoRangeHigh 删除异常值数量: 3569
openAcc 删除异常值数量: 4998
pubRec 删除异常值数量: 4081
revolBal 删除异常值数量: 5360
revolUtil 删除异常值数量: 23
totalAcc 删除异常值数量: 3686
earliesCreditLine 删除异常值数量: 4141
title 删除异常值数量: 13202
n0 删除异常值数量: 8046
n1 删除异常值数量: 5462
n2 删除异常值数量: 3856
n3 删除异常值数量: 2767
n4 删除异常值数量: 3174
n5 删除异常值数量: 3955
n6 删除异常值数量: 6266
n7 删除异常值数量: 2511
n8 删除异常值数量: 2506
n9 删除异常值数量: 119
n10 删除异常值数量: 2305
n13 删除异常值数量: 13738
n14 删除异常值数量: 5423
issueDateDT 删除异常值数量: 0
3-sigma 删除后维度: (323837, 46)


In [41]:
# =========================
# 14. 删除 issueDate 列
# =========================

if "issueDate" in df.columns:
    df = df.drop(columns=["issueDate"])

print("删除 issueDate 后维度:", df.shape)

删除 issueDate 后维度: (323837, 45)


In [45]:
# =========================
# 15. 最终检查数据集
# =========================

print("最终数据维度:", df.shape)

print("剩余缺失值数量:")
print(df.isnull().sum().sum())

print("非数值型变量:")
print(df.select_dtypes(exclude=np.number).columns.tolist())

print("列名:")
print(df.columns.tolist())

df.head().T

最终数据维度: (323837, 45)
剩余缺失值数量:
0
非数值型变量:
[]
列名:
['loanAmnt', 'term', 'interestRate', 'installment', 'grade', 'subGrade', 'employmentTitle', 'employmentLength', 'homeOwnership', 'annualIncome', 'verificationStatus', 'isDefault', 'purpose', 'postCode', 'regionCode', 'dti', 'delinquency_2years', 'ficoRangeLow', 'ficoRangeHigh', 'openAcc', 'pubRec', 'pubRecBankruptcies', 'revolBal', 'revolUtil', 'totalAcc', 'initialListStatus', 'applicationType', 'earliesCreditLine', 'title', 'n0', 'n1', 'n2', 'n3', 'n4', 'n5', 'n6', 'n7', 'n8', 'n9', 'n10', 'n11', 'n12', 'n13', 'n14', 'issueDateDT']


,0,1,2,3,4
loanAmnt,35000.00,12000.00,2050.00,11500.00,24000.00
term,5.00,5.00,3.00,3.00,3.00
interestRate,19.52,16.99,7.69,14.98,9.99
installment,917.97,298.17,63.95,398.54,774.30
grade,4.00,3.00,0.00,2.00,1.00
subGrade,21.00,17.00,3.00,12.00,7.00
employmentTitle,320.00,31698.00,180083.00,214017.00,4967.00
employmentLength,2.00,8.00,9.00,1.00,10.00
homeOwnership,2.00,0.00,0.00,1.00,0.00
annualIncome,110000.00,74000.00,35000.00,30000.00,150000.00


In [47]:
# =========================
# 15. 最终检查数据集
# =========================

print("最终数据维度:", df.shape)

print("剩余缺失值数量:")
print(df.isnull().sum().sum())

print("非数值型变量:")
print(df.select_dtypes(exclude=np.number).columns.tolist())

print("列名:")
print(df.columns.tolist())

df.head()

最终数据维度: (323837, 45)
剩余缺失值数量:
0
非数值型变量:
[]
列名:
['loanAmnt', 'term', 'interestRate', 'installment', 'grade', 'subGrade', 'employmentTitle', 'employmentLength', 'homeOwnership', 'annualIncome', 'verificationStatus', 'isDefault', 'purpose', 'postCode', 'regionCode', 'dti', 'delinquency_2years', 'ficoRangeLow', 'ficoRangeHigh', 'openAcc', 'pubRec', 'pubRecBankruptcies', 'revolBal', 'revolUtil', 'totalAcc', 'initialListStatus', 'applicationType', 'earliesCreditLine', 'title', 'n0', 'n1', 'n2', 'n3', 'n4', 'n5', 'n6', 'n7', 'n8', 'n9', 'n10', 'n11', 'n12', 'n13', 'n14', 'issueDateDT']


,loanAmnt,term,interestRate,installment,grade,subGrade,employmentTitle,employmentLength,homeOwnership,annualIncome,...,n6,n7,n8,n9,n10,n11,n12,n13,n14,issueDateDT
0,35000.0,5,19.52,917.97,4,21,320.0,2,2,110000.0,...,8.0,4.0,12.0,2.0,7.0,0.0,0.0,0.0,2.0,2587
1,12000.0,5,16.99,298.17,3,17,31698.0,8,0,74000.0,...,21.0,4.0,5.0,3.0,11.0,0.0,0.0,0.0,4.0,3044
2,2050.0,3,7.69,63.95,0,3,180083.0,9,0,35000.0,...,3.0,10.0,18.0,3.0,12.0,0.0,0.0,0.0,3.0,2679
3,11500.0,3,14.98,398.54,2,12,214017.0,1,1,30000.0,...,10.0,5.0,21.0,4.0,8.0,0.0,0.0,0.0,2.0,2406
4,24000.0,3,9.99,774.30,1,7,4967.0,10,0,150000.0,...,7.0,6.0,17.0,3.0,7.0,0.0,0.0,0.0,2.0,2983


In [49]:


df.to_csv("train_clean.csv", index=False)

print("保存成功: train_clean.csv")

保存成功: train_clean.csv
